# DEBE: Dengue Triage Agent¶
* **Competition: MedGemma Impact Challenge — Agentic Workflow**
*  Track Model : google/medgemma-4b-it + arumpuri/medgemma-4b-debe-specialist (LoRA)

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

###  INSTALLATION   

In [2]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 1 — INSTALLATION                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Run this cell first. It installs every dependency needed for the notebook.
# Estimasi waktu: ~2 menit pada GPU T4.

!pip install -q -U \
    "pillow<11.0" \
    "transformers>=4.49.0" \
    accelerate \
    bitsandbytes \
    gradio \
    huggingface_hub \
    peft


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


###  IMPORTS & GPU CONFIGURATION 

In [3]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 2 — IMPORTS & GPU CONFIGURATION                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import torch
import numpy as np
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
from huggingface_hub import login
from peft import PeftModel
import gradio as gr
import warnings
import contextlib
import time
import re

# Stability flags for Kaggle T4 GPU
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)
warnings.filterwarnings("ignore")

print("=" * 80)
print("  DeBe V3.0 — Glass Box Bilingual Triage Engine")
print("  Powered by MedGemma 4B-IT + DeBe LoRA Specialist")
print("=" * 80)


  DeBe V3.0 — Glass Box Bilingual Triage Engine
  Powered by MedGemma 4B-IT + DeBe LoRA Specialist


###  AUTHENTICATION
How to set up your token:

1. Go to https://huggingface.co/settings/tokens
2. Create a READ token
3. In Kaggle: Add-ons → Secrets → Add a new secret named HF_TOKEN


In [4]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 3 — AUTHENTICATION                                  
# ╚══════════════════════════════════════════════════════════════════════════════╝
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)


 ### MODEL LOADING                                                   
                                                                              
We load the base MedGemma 4B-IT in 4-bit NF4 quantization, then inject DeBe LoRA adapter, which specialises the model for DHF triage using Indonesian clinical datasets.                                               
                                                                              
Model yang digunakan:                                                       
* Base : google/medgemma-4b-it (Multimodal, 4-bit NF4)
* LoRA : arumpuri/medgemma-4b-debe-specialist (~119 MB adapter)            


In [5]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 4 — MODEL LOADING                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("\n[4/9] Loading Base Model & DeBe LoRA Adapter ...")

BASE_MODEL_ID  = "google/medgemma-4b-it"
ADAPTER_ID     = "arumpuri/medgemma-4b-debe-specialist"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)

print(f"  → Injecting LoRA: {ADAPTER_ID}")
model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
print("  ✓ DeBe Specialist Model loaded successfully!\n")




[4/9] Loading Base Model & DeBe LoRA Adapter ...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

  → Injecting LoRA: arumpuri/medgemma-4b-debe-specialist


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/119M [00:00<?, ?B/s]

  ✓ DeBe Specialist Model loaded successfully!



### CORE INFERENCE ENGINE  

In [6]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 5 — CORE INFERENCE ENGINE                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def query_medgemma(
    prompt: str,
    image: "Image.Image | None" = None,
    max_tokens: int = 400,
    use_adapter: bool = True,
) -> str:
    """
    Single inference call to MedGemma.
    • image      : optional PIL image (Vision Agent only)
    • use_adapter: False → uses base MedGemma (better for raw image tasks)
    """
    try:
        content = []
        if image is not None:
            content.append({"type": "image", "image": image})
        content.append({"type": "text", "text": prompt})

        messages = [{"role": "user", "content": content}]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        ctx = contextlib.nullcontext() if use_adapter else model.disable_adapter()
        with ctx, torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=False,
                num_beams=1,
                repetition_penalty=1.05,
            )

        input_len = inputs["input_ids"].shape[-1]
        return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

    except Exception as exc:
        return f"**[Error / Kesalahan]** {exc}"


def safe_parse(text: str, marker: str, fallback: str) -> str:
    """Extract text after a known marker; returns fallback if marker not found."""
    idx = text.find(marker)
    if idx == -1:
        return fallback
    return text[idx:].strip()

###  DETERMINISTIC RISK SCORING (Neuro-Symbolic Layer)              
                                                                             
Python handles all arithmetic so the LLM never has to do maths. This makes the risk score 100% reproducible and auditable.                 


In [7]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 6 — DETERMINISTIC RISK SCORING (Neuro-Symbolic Layer)                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def sanitize(value, default=0.0):
    try:
        if value is None or str(value).strip() == "":
            return float(default)
        return float(value)
    except Exception:
        return float(default)


def calculate_who_risk(data: dict) -> tuple[int, str, str]:
    """
    Returns (score, label_en, label_id).
    Score ranges: LOW 0-2, MODERATE 3-4, HIGH 5-7, CRITICAL 8+
    """
    score = 0
    plt = sanitize(data.get("platelets"), 0)
    if plt > 0:
        if plt < 20_000:   score += 4
        elif plt < 50_000: score += 3
        elif plt < 100_000:score += 2
        elif plt < 150_000:score += 1

    score += int(sanitize(data.get("warning_count"), 0))

    if score >= 8:   return score, "CRITICAL",  "KRITIS"
    elif score >= 5: return score, "HIGH",       "TINGGI"
    elif score >= 3: return score, "MODERATE",   "SEDANG"
    return score,    "LOW",        "RENDAH"


RISK_STYLES = {
    "CRITICAL": {"bg": "#fef2f2", "border": "#dc2626", "text": "#991b1b", "icon": "🔴"},
    "HIGH":     {"bg": "#fff7ed", "border": "#ea580c", "text": "#9a3412", "icon": "🟠"},
    "MODERATE": {"bg": "#fefce8", "border": "#ca8a04", "text": "#854d0e", "icon": "🟡"},
    "LOW":      {"bg": "#f0fdf4", "border": "#16a34a", "text": "#14532d", "icon": "🟢"},
}


def risk_badge_html(score: int, label_en: str, label_id: str) -> str:
    s = RISK_STYLES.get(label_en, {"bg": "#f3f4f6", "border": "#6b7280", "text": "#374151", "icon": "⚪"})
    return (
        f'<div style="'
        f'display:flex;align-items:center;gap:14px;'
        f'padding:16px 24px;border-radius:12px;'
        f'background:{s["bg"]};'
        f'border:2px solid {s["border"]};'
        f'margin:12px 0 20px 0;'
        f'font-family:Plus Jakarta Sans,system-ui,sans-serif;'
        f'">'
        f'<span style="font-size:2rem;">{s["icon"]}</span>'
        f'<div>'
        f'<div style="font-size:0.7rem;font-weight:700;letter-spacing:0.1em;'
        f'text-transform:uppercase;color:{s["text"]};opacity:0.7;">WHO Risk Score</div>'
        f'<div style="font-size:1.25rem;font-weight:800;color:{s["text"]};">'
        f'{label_en} / {label_id} &nbsp;<span style="font-weight:400;font-size:1rem;">({score}/15)</span>'
        f'</div>'
        f'</div>'
        f'</div>'
    )



### MULTI-AGENT SYSTEM                                              
                                                                             
Four specialist agents collaborate in sequence:                             
* Agent 1 Vision   → detects petechiae / rash from skin photo
* Agent 2 Symptom  → maps clinical phase using day-of-illness + pain
* Agent 3 Lab      → interprets platelet / hematocrit flags
* Agent 4 Decision → synthesises all evidence into a triage decision                                                                                    

In [8]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 7 — MULTI-AGENT SYSTEM                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

BILINGUAL_SUFFIX = (
    "\n\nOUTPUT RULES — follow exactly, no exceptions:\n"
    "Write your response using ONLY these two XML tags:\n"
    "<EN>your full answer in English here</EN>\n"
    "<ID>terjemahan lengkap dalam Bahasa Indonesia di sini</ID>\n"
    "Do NOT write anything outside these tags. Both tags are mandatory."
)


def parse_bilingual(raw: str) -> tuple[str, str]:
    """
    Robustly extract EN and ID sections from model output.
    Strategy (tried in order):
      1. XML tags  <EN>...</EN>  <ID>...</ID>   — the intended format
      2. Labelled headers  EN:  /  ID:  on their own line
      3. The literal string '---' as a divider
      4. Last-resort: treat the whole output as English, flag ID as missing
    """
    import re

    # ── Strategy 1: XML tags ────────────────────────────────────────────────
    en_match = re.search(r"<EN>(.*?)</EN>", raw, re.DOTALL | re.IGNORECASE)
    id_match = re.search(r"<ID>(.*?)</ID>", raw, re.DOTALL | re.IGNORECASE)
    if en_match and id_match:
        return en_match.group(1).strip(), id_match.group(1).strip()

    # ── Strategy 2: Labelled header lines  (EN: / ID:) ─────────────────────
    en_hdr = re.search(r"(?:^|\n)\s*EN\s*:\s*(.*?)(?=\n\s*ID\s*:|\Z)", raw, re.DOTALL | re.IGNORECASE)
    id_hdr = re.search(r"(?:^|\n)\s*ID\s*:\s*(.*?)$", raw, re.DOTALL | re.IGNORECASE)
    if en_hdr and id_hdr:
        return en_hdr.group(1).strip(), id_hdr.group(1).strip()

    # ── Strategy 3: '---' divider ───────────────────────────────────────────
    if "---" in raw:
        parts = raw.split("---", 1)
        return parts[0].strip(), parts[1].strip()

    # ── Strategy 4: Graceful fallback ───────────────────────────────────────
    return raw.strip(), "_(Terjemahan tidak dihasilkan — model tidak mengikuti format output. Silakan jalankan ulang.)_"


class VisionAgent:
    """
    Analyses a skin photograph for Dengue warning signs (petechiae, rash).
    Uses BASE MedGemma (no LoRA) because the base model has stronger vision priors.

    Menganalisis foto kulit untuk tanda bahaya Dengue menggunakan MedGemma dasar.
    """

    def analyze(self, image: "Image.Image | None", data: dict) -> str:
        if not image or not isinstance(image, Image.Image): # <-- DEFENSIVE CHECK
            return (
                "**🔍 Visual Assessment (EN):**\n"
                "No skin photograph was uploaded. Visual assessment bypassed. "
                "Triage relies entirely on clinical and laboratory parameters.\n\n"
                "---\n\n"
                "**🔍 Penilaian Visual (ID):**\n"
                "Tidak ada foto kulit yang diunggah. Penilaian visual dilewati. "
                "Triase sepenuhnya bergantung pada parameter klinis dan laboratorium."
            )

        prompt = (
            f"You are a dermatologist reviewing a skin photograph.\n"
            f"Patient context: Day {data['day_of_illness']} of illness, "
            f"temperature {data['fever_celsius']}°C.\n\n"
            f"Task: Identify any signs of Dengue Hemorrhagic Fever, specifically:\n"
            f"• Petechiae (pinpoint red/purple bleeding spots)\n"
            f"• Confluent rash or flushing\n"
            f"• Bruising or purpura\n\n"
            f"State clearly whether this is a Dengue Warning Sign: Yes / No / Unclear.\n"
            f"{BILINGUAL_SUFFIX}"
        )
        raw = query_medgemma(prompt, image=image, max_tokens=300, use_adapter=False)
        return self._format(raw)

    @staticmethod
    def _format(raw: str) -> str:
        en, id_ = parse_bilingual(raw)
        return (
            f"**🔍 Visual Assessment (EN):**\n{en}\n\n"
            f"---\n\n"
            f"**🔍 Penilaian Visual (ID):**\n{id_}"
        )


class SymptomAgent:
    """
    Maps the patient's symptom profile to the standard WHO Dengue clinical phases.

    Memetakan profil gejala pasien ke fase klinis Dengue WHO.
    """

    def analyze(self, data: dict, vision_summary: str) -> str:
        prompt = (
            f"You are a tropical medicine specialist.\n\n"
            f"PATIENT DATA:\n"
            f"• Day of Illness  : {data['day_of_illness']}\n"
            f"• Temperature     : {data['fever_celsius']}°C\n"
            f"• Headache        : {data['headache_severity']}/10\n"
            f"• Myalgia/Arthralgia: {data['myalgia_severity']}/10\n"
            f"• Visual Findings : {vision_summary[:200]}\n\n"
            f"Task: Determine the WHO Dengue Clinical Phase.\n"
            f"Phases — Febrile: Day 1-3 | Critical: Day 4-6 | Recovery: Day 7+\n"
            f"Justify your phase classification using the data above.\n"
            f"{BILINGUAL_SUFFIX}"
        )
        raw = query_medgemma(prompt, image=None, max_tokens=500, use_adapter=True)
        return self._format(raw)

    @staticmethod
    def _format(raw: str) -> str:
        en, id_ = parse_bilingual(raw)
        return (
            f"**🩺 Clinical Phase Assessment (EN):**\n{en}\n\n"
            f"---\n\n"
            f"**🩺 Penilaian Fase Klinis (ID):**\n{id_}"
        )


class LabAgent:
    """
    Interprets haematology results using pre-computed symbolic flags.
    The LLM explains clinical significance; Python did the arithmetic.

    Menginterpretasi hasil hematologi menggunakan flag simbolik yang telah dihitung.
    """

    def analyze(self, data: dict, risk_score: int, risk_label_en: str, risk_label_id: str) -> str:
        plt = sanitize(data.get("platelets"), 0)
        hct = sanitize(data.get("hct"), 0)

        if plt == 0 and hct == 0:
            return (
                "**🩸 Laboratory Assessment (EN):**\n"
                "Lab data not available. Urgent Full Blood Count (FBC/CBC) is strongly "
                "recommended. Decision based on clinical parameters only.\n\n"
                "---\n\n"
                "**🩸 Penilaian Laboratorium (ID):**\n"
                "Data laboratorium tidak tersedia. Pemeriksaan Darah Lengkap (FBC/CBC) "
                "sangat disarankan segera. Keputusan berdasarkan parameter klinis saja."
            )

        plt_flag = "CRITICAL — severe thrombocytopenia (<100k)" if plt < 100_000 else \
                   "BORDERLINE — monitor closely (<150k)"       if plt < 150_000 else "NORMAL"
        hct_flag = "ELEVATED — plasma leakage suspected (>45%)" if hct > 45.0 else "NORMAL"

        prompt = (
            f"You are a haematologist reviewing Dengue lab results.\n\n"
            f"PRE-COMPUTED FLAGS (calculated by system, trust these values):\n"
            f"• Platelets : {int(plt):,} /μL  →  {plt_flag}\n"
            f"• Hematocrit: {hct:.1f}%         →  {hct_flag}\n"
            f"• WHO Risk Score: {risk_label_en} ({risk_score}/15)\n\n"
            f"Task: Explain the clinical danger of these flags in Dengue Hemorrhagic Fever. "
            f"State what immediate monitoring or intervention is warranted.\n"
            f"{BILINGUAL_SUFFIX}"
        )
        raw = query_medgemma(prompt, image=None, max_tokens=500, use_adapter=True)
        return self._format(raw)

    @staticmethod
    def _format(raw: str) -> str:
        en, id_ = parse_bilingual(raw)
        return (
            f"**🩸 Laboratory Assessment (EN):**\n{en}\n\n"
            f"---\n\n"
            f"**🩸 Penilaian Laboratorium (ID):**\n{id_}"
        )


class DecisionAgent:
    """
    Senior Triage Orchestrator: synthesises all agent evidence into a final
    WHO-aligned triage decision with bilingual disposition and action plan.

    Orkestrator triase senior: mensintesis semua bukti menjadi keputusan triase
    sesuai WHO dengan disposisi dan rencana tindakan bilingual.
    """

    def decide(
        self,
        data: dict,
        vision_log: str,
        sym_log: str,
        lab_log: str,
        risk_score: int,
        risk_label_en: str,
        risk_label_id: str,
        badge_html: str,
    ) -> str:
        prompt = (
            f"You are a senior emergency physician synthesising a Dengue triage report.\n\n"
            f"AGENT SUMMARIES:\n"
            f"[Visual]  : {vision_log[:250]}\n"
            f"[Symptom] : {sym_log[:250]}\n"
            f"[Lab]     : {lab_log[:250]}\n"
            f"[WHO Risk]: {risk_label_en} ({risk_score}/15)\n\n"
            f"Task: Produce a final triage decision using the WHO 2009 Dengue Classification:\n"
            f"  • Suspected Dengue (no warning signs) → Home care with monitoring\n"
            f"  • Dengue with Warning Signs            → Hospital referral\n"
            f"  • Severe Dengue                        → Emergency hospitalisation\n\n"
            f"Your output MUST include:\n"
            f"  1. Classification (one of the three above)\n"
            f"  2. Immediate action\n"
            f"  3. Follow-up instructions for the family / caregiver\n"
            f"{BILINGUAL_SUFFIX}"
        )
        raw = query_medgemma(prompt, image=None, max_tokens=900, use_adapter=True)
        en, id_ = parse_bilingual(raw)

        return (
            f"{badge_html}\n\n"
            f"---\n\n"
            f"### 🏥 Final Triage Decision (EN)\n\n{en}\n\n"
            f"---\n\n"
            f"### 🏥 Keputusan Triase Akhir (ID)\n\n{id_}"
        )


# Instantiate agents once at module level
vision_agent  = VisionAgent()
symptom_agent = SymptomAgent()
lab_agent     = LabAgent()
decision_agent = DecisionAgent()


###  GRADIO ORCHESTRATOR

In [9]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 8 — GRADIO ORCHESTRATOR (Streaming UI)                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

PENDING_MSG = (
    "⏳ *Menunggu / Waiting for previous agent to finish...*"
)
INIT_MSG = "🔄 *Menginisialisasi alur kerja / Initializing workflow...*"


def gradio_wrapper(
    age, gender, day, fever, headache, myalgia,
    platelets, hct, wbc, warnings, rash_image
):
    data = {
        "age"             : sanitize(age, 25),
        "gender"          : gender or "Female",
        "day_of_illness"  : sanitize(day, 1),
        "fever_celsius"   : sanitize(fever, 37.0),
        "headache_severity": sanitize(headache, 0),
        "myalgia_severity" : sanitize(myalgia, 0),
        "platelets"       : sanitize(platelets, 0),
        "hct"             : sanitize(hct, 0.0),
        "wbc"             : sanitize(wbc, 0),
        "warning_count"   : sanitize(warnings, 0),
    }

    # ── Step 0: Init ──────────────────────────────────────────────────────────
    yield INIT_MSG, PENDING_MSG, PENDING_MSG, PENDING_MSG

    # ── Step 1: Vision ────────────────────────────────────────────────────────
    label1 = "🔍 *Agen Visual sedang menganalisis / Vision Agent analyzing...*"
    yield label1, PENDING_MSG, PENDING_MSG, PENDING_MSG
    vis_out = vision_agent.analyze(rash_image, data)
    torch.cuda.empty_cache()

    # ── Step 2: Symptom ───────────────────────────────────────────────────────
    label2 = "🩺 *Agen Gejala memetakan fase / Symptom Agent mapping phase...*"
    yield vis_out, label2, PENDING_MSG, PENDING_MSG
    sym_out = symptom_agent.analyze(data, vis_out)
    torch.cuda.empty_cache()

    # ── Step 3: Lab ───────────────────────────────────────────────────────────
    label3 = "🧪 *Agen Lab memeriksa hematologi / Lab Agent checking labs...*"
    yield vis_out, sym_out, label3, PENDING_MSG
    risk_score, risk_label_en, risk_label_id = calculate_who_risk(data)
    lab_out = lab_agent.analyze(data, risk_score, risk_label_en, risk_label_id)
    torch.cuda.empty_cache()

    # ── Step 4: Decision ──────────────────────────────────────────────────────
    label4 = "⚖️ *Orkestrator mensintesis keputusan / Orchestrator synthesising...*"
    yield vis_out, sym_out, lab_out, label4
    badge = risk_badge_html(risk_score, risk_label_en, risk_label_id)
    final_out = decision_agent.decide(
        data, vis_out, sym_out, lab_out,
        risk_score, risk_label_en, risk_label_id, badge
    )
    torch.cuda.empty_cache()

    yield vis_out, sym_out, lab_out, final_out


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 9 — GRADIO UI                                                       ║
# ║                                                                              ║
# ║  Clinical aesthetic: dark teal + amber accent, bilingual labels everywhere. ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Custom CSS ────────────────────────────────────────────────────────────────
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&family=Sora:wght@700;800&family=JetBrains+Mono:wght@400;600&display=swap');

/* ── Global Reset to Light Theme ── */
:root {
    --teal-700:  #0f766e;
    --teal-600:  #0d9488;
    --teal-500:  #14b8a6;
    --teal-100:  #ccfbf1;
    --teal-50:   #f0fdfa;
    --green-600: #16a34a;
    --amber-500: #f59e0b;
    --amber-100: #fef3c7;
    --red-600:   #dc2626;
    --red-50:    #fef2f2;
    --orange-500:#ea580c;
    --orange-50: #fff7ed;
    --gray-900:  #111827;
    --gray-700:  #374151;
    --gray-500:  #6b7280;
    --gray-300:  #d1d5db;
    --gray-100:  #f3f4f6;
    --gray-50:   #f9fafb;
    --white:     #ffffff;
    --font-display: 'Sora', system-ui, sans-serif;
    --font-body:    'Plus Jakarta Sans', system-ui, sans-serif;
    --font-mono:    'JetBrains Mono', monospace;
    --shadow-sm: 0 1px 3px rgba(0,0,0,0.08), 0 1px 2px rgba(0,0,0,0.06);
    --shadow-md: 0 4px 12px rgba(0,0,0,0.10), 0 2px 4px rgba(0,0,0,0.06);
    --shadow-lg: 0 10px 30px rgba(0,0,0,0.12), 0 4px 8px rgba(0,0,0,0.06);
    --radius:    12px;
}

/* Force light background on every Gradio container */
body,
.gradio-container,
.gradio-container > *,
.contain,
#component-0,
.gap,
.form {
    background: var(--gray-50) !important;
    color: var(--gray-900) !important;
    font-family: var(--font-body) !important;
}

/* ── Hero Header ── */
.debe-header {
    background: linear-gradient(135deg, var(--teal-700) 0%, #134e4a 60%, #0f3d3a 100%);
    border-radius: 16px;
    padding: 36px 44px;
    margin-bottom: 28px;
    position: relative;
    overflow: hidden;
    box-shadow: var(--shadow-lg);
}
.debe-header::after {
    content: '🦟';
    position: absolute;
    right: 40px; top: 50%;
    transform: translateY(-50%);
    font-size: 7rem;
    opacity: 0.12;
    pointer-events: none;
}
.debe-header h1 {
    font-family: var(--font-display) !important;
    font-size: 2.4rem !important;
    font-weight: 800 !important;
    color: #ffffff !important;
    margin: 0 0 8px 0 !important;
    letter-spacing: -0.02em;
    line-height: 1.1;
}
.debe-header .subtitle {
    color: #99f6e4;
    font-size: 0.95rem;
    line-height: 1.7;
    max-width: 640px;
}
.debe-header .model-badge {
    display: inline-block;
    background: rgba(255,255,255,0.15);
    border: 1px solid rgba(255,255,255,0.3);
    color: #ffffff;
    font-family: var(--font-mono);
    font-size: 0.72rem;
    padding: 4px 12px;
    border-radius: 20px;
    margin-top: 12px;
    backdrop-filter: blur(4px);
}

/* ── Section Labels ── */
.section-label {
    display: flex;
    align-items: center;
    gap: 8px;
    font-family: var(--font-mono);
    font-size: 0.68rem;
    font-weight: 600;
    letter-spacing: 0.14em;
    text-transform: uppercase;
    color: var(--teal-700);
    background: var(--teal-50);
    border: 1px solid var(--teal-100);
    border-radius: 8px;
    padding: 8px 14px;
    margin-bottom: 16px;
}

/* ── Cards for input groups ── */
.card {
    background: var(--white) !important;
    border: 1px solid var(--gray-300) !important;
    border-radius: var(--radius) !important;
    padding: 20px !important;
    margin-bottom: 16px !important;
    box-shadow: var(--shadow-sm) !important;
}

/* ── All Gradio inputs — force readable light style ── */
input[type="number"],
input[type="text"],
textarea,
select,
.gr-input,
.gr-textarea,
.gr-box {
    background: var(--white) !important;
    color: var(--gray-900) !important;
    border: 1.5px solid var(--gray-300) !important;
    border-radius: 8px !important;
    font-family: var(--font-body) !important;
    font-size: 0.9rem !important;
}
input[type="number"]:focus,
input[type="text"]:focus,
textarea:focus {
    border-color: var(--teal-500) !important;
    box-shadow: 0 0 0 3px rgba(20,184,166,0.15) !important;
    outline: none !important;
}

/* ── Gradio labels ── */
label, .gr-form label, span.svelte-1ed2p3z,
.label-wrap span, .block span.svelte-1gfkn6j {
    color: var(--gray-700) !important;
    font-family: var(--font-body) !important;
    font-size: 0.84rem !important;
    font-weight: 600 !important;
}

/* ── Info/hint text ── */
.info, .gr-info, span.info {
    color: var(--gray-500) !important;
    font-size: 0.78rem !important;
}

/* ── Accordion headers ── */
.gr-accordion > button,
details > summary,
.label-wrap {
    background: var(--white) !important;
    color: var(--gray-900) !important;
    font-family: var(--font-body) !important;
    font-weight: 600 !important;
    border-radius: 10px !important;
    border: 1px solid var(--gray-300) !important;
}

/* ── Accordion body ── */
.gr-accordion .wrap,
details[open] > div {
    background: var(--gray-50) !important;
    border: 1px solid var(--gray-300) !important;
    border-top: none !important;
    border-radius: 0 0 10px 10px !important;
}

/* ── Output Markdown panels ── */
.output-panel .prose,
.gr-markdown,
.markdown-body,
.prose {
    background: var(--white) !important;
    color: var(--gray-900) !important;
    padding: 16px !important;
    border-radius: 10px !important;
    border: 1px solid var(--gray-200) !important;
    font-family: var(--font-body) !important;
    font-size: 0.9rem !important;
    line-height: 1.75 !important;
}

/* ── Submit Button ── */
.submit-btn,
.submit-btn button {
    background: linear-gradient(135deg, var(--teal-700) 0%, var(--teal-600) 100%) !important;
    color: #ffffff !important;
    font-family: var(--font-body) !important;
    font-weight: 700 !important;
    font-size: 1rem !important;
    border: none !important;
    border-radius: 10px !important;
    padding: 14px 24px !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
    box-shadow: 0 4px 16px rgba(15,118,110,0.35) !important;
    width: 100% !important;
}
.submit-btn:hover,
.submit-btn button:hover {
    background: linear-gradient(135deg, #0c6660 0%, var(--teal-700) 100%) !important;
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 24px rgba(15,118,110,0.45) !important;
}

/* ── Examples table ── */
.gr-samples-table,
.gr-samples,
table {
    background: var(--white) !important;
    color: var(--gray-900) !important;
    border-radius: 8px !important;
    overflow: hidden !important;
}
.gr-samples-table td, .gr-samples-table th,
table td, table th {
    color: var(--gray-900) !important;
    background: var(--white) !important;
    border-color: var(--gray-200) !important;
    font-size: 0.82rem !important;
}
.gr-samples-table tr:hover td,
table tr:hover td {
    background: var(--teal-50) !important;
}

/* ── Slider track ── */
input[type="range"] {
    accent-color: var(--teal-600) !important;
}

/* ── Dropdown ── */
.gr-dropdown, select {
    background: var(--white) !important;
    color: var(--gray-900) !important;
}

/* ── Image upload area ── */
.gr-image, .image-container, .upload-container {
    background: var(--gray-50) !important;
    border: 2px dashed var(--teal-500) !important;
    border-radius: 10px !important;
    color: var(--gray-700) !important;
}

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: var(--gray-100); }
::-webkit-scrollbar-thumb { background: var(--teal-500); border-radius: 3px; }

/* ── Gradio footer / branding ── */
.built-with, footer {
    display: none !important;
}

/* ── Pending state styling ── */
em {
    color: var(--gray-500) !important;
}
"""

# ── Header HTML ───────────────────────────────────────────────────────────────
HEADER_HTML = """
<div class="debe-header">
    <h1>🦟 DeBe Triage Engine</h1>
    <p class="subtitle">
        <strong>Demam Berdarah (DHF) Multi-Agent Triage &amp; Decision Support</strong><br>
        Glass-Box AI for Indonesian Puskesmas — Results in English &amp; Bahasa Indonesia<br>
        AI Transparan untuk Puskesmas Indonesia — Hasil dalam Bahasa Inggris &amp; Indonesia
    </p>
    <span class="model-badge">medgemma-4b-it + arumpuri/medgemma-4b-debe-specialist (LoRA)</span>
</div>
"""

# ── How to Use (Bilingual) ────────────────────────────────────────────────────
HOW_TO_USE = """
## 📖 How to Use / Cara Penggunaan

**English:**
1. Fill in the patient's **vitals and demographics** in Column 1.
2. Enter any available **lab results** (leave `0` if not yet available — the system handles missing data gracefully).
3. Optionally upload a **skin photo** to enable the Visual Agent (petechiae detection).
4. Click **Run Multi-Agent Triage** and watch each agent reason step-by-step.
5. Each output panel shows results in **English** then **Indonesian**.

**Bahasa Indonesia:**
1. Isi **tanda-tanda vital dan data demografis** pasien di Kolom 1.
2. Masukkan **hasil laboratorium** yang tersedia (isi `0` jika belum ada — sistem menangani data yang hilang).
3. Opsional: unggah **foto kulit** untuk mengaktifkan Agen Visual (deteksi petekie).
4. Klik **Run Multi-Agent Triage** dan amati setiap agen memberikan alasan langkah demi langkah.
5. Setiap panel output menampilkan hasil dalam **Bahasa Inggris** lalu **Bahasa Indonesia**.

> ⚠️ **Disclaimer / Penafian:** This tool is a clinical decision *support* system for trained healthcare workers — not a replacement for physician judgement.
> Alat ini adalah sistem *pendukung* keputusan klinis untuk tenaga kesehatan terlatih — bukan pengganti penilaian dokter.
"""

# ── Architecture Note ─────────────────────────────────────────────────────────
ARCH_NOTE = """
### 🏗️ System Architecture / Arsitektur Sistem

```
Patient Input ──→ [Vision Agent]   ──→ skin photo analysis
                  [Symptom Agent]  ──→ clinical phase mapping
                  [Lab Agent]      ──→ haematology interpretation   ──→ [Orchestrator]
                  [Risk Scorer]    ──→ deterministic WHO score (Python)          │
                                                                                  ↓
                                                               Final Bilingual Triage Report
```

- **Vision Agent** uses **base MedGemma** (stronger vision priors, no LoRA)
- **Symptom / Lab / Decision Agents** use **DeBe LoRA adapter** (fine-tuned on Indonesian DHF data)
- **Risk Score** is computed deterministically by Python — LLM only *explains*, never *calculates*
"""



# ── Build Gradio App ──────────────────────────────────────────────────────────
with gr.Blocks(
    css=CUSTOM_CSS,
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="emerald",
        neutral_hue="gray",
        font=[gr.themes.GoogleFont("Plus Jakarta Sans"), "ui-sans-serif", "system-ui"],
    ),
    title="DeBe — Demam Berdarah Triage Engine",
) as iface:

    gr.HTML(HEADER_HTML)
    gr.Markdown(HOW_TO_USE)
    gr.Markdown(ARCH_NOTE)

    with gr.Row(equal_height=False):

        # ── LEFT: Input Panel ──────────────────────────────────────────────────
        with gr.Column(scale=4):
            gr.HTML('<div class="section-label">📋 Section 1 — Patient Data / Data Pasien</div>')

            with gr.Row():
                age = gr.Number(
                    label="Age / Usia (years/tahun)",
                    value=25, minimum=0, maximum=120,
                )
                gender = gr.Dropdown(
                    choices=["Male / Laki-laki", "Female / Perempuan"],
                    label="Gender / Jenis Kelamin",
                    value="Female / Perempuan",
                )

            with gr.Row():
                day = gr.Number(
                    label="Day of Illness / Hari Sakit",
                    value=4, minimum=1, maximum=14,
                    info="Critical phase = Day 4–6 / Fase kritis = Hari 4–6",
                )
                fever = gr.Number(
                    label="Temperature / Suhu (°C)",
                    value=38.5, minimum=35.0, maximum=42.0,
                )

            with gr.Accordion(
                "Pain Levels / Tingkat Nyeri (optional / opsional)", open=False
            ):
                headache = gr.Slider(
                    0, 10, value=7, step=1,
                    label="Headache / Sakit Kepala (0–10)",
                )
                myalgia = gr.Slider(
                    0, 10, value=8, step=1,
                    label="Myalgia / Nyeri Otot-Sendi (0–10)",
                )

            gr.HTML('<div class="section-label" style="margin-top:16px;">🩸 Section 2 — Laboratory Data / Data Laboratorium</div>')
            gr.Markdown(
                "*Leave `0` if unavailable — the system handles missing data. / "
                "Biarkan `0` jika belum tersedia — sistem menangani data yang hilang.*"
            )

            with gr.Row():
                platelets = gr.Number(
                    label="Platelets / Trombosit (/μL)",
                    value=0,
                    info="Normal: >150,000 | Warning: <100,000",
                )
                hct = gr.Number(
                    label="Hematocrit / Hematokrit (%)",
                    value=0.0,
                    info="Plasma leakage risk / Risiko kebocoran plasma: >45%",
                )

            with gr.Row():
                wbc = gr.Number(
                    label="WBC / Leukosit (/μL)",
                    value=0,
                )
                warnings_inp = gr.Number(
                    label="Warning Signs Count / Jumlah Tanda Bahaya",
                    value=0,
                    info="e.g. vomiting, mucosal bleed / mis. muntah, perdarahan mukosa",
                )

            gr.HTML('<div class="section-label" style="margin-top:16px;">📸 Section 3 — Visual Evidence / Bukti Visual</div>')
            img = gr.Image(
                type="pil",
                label="Skin Photo for Petechiae Detection / Foto Kulit untuk Deteksi Petekie (optional / opsional)",
            )

            submit_btn = gr.Button(
                "🚀 Run Multi-Agent Triage / Jalankan Triase Multi-Agen",
                variant="primary",
                size="lg",
                elem_classes=["submit-btn"],
            )

            gr.Markdown("---")
            gr.Markdown("### 🧪 Quick Test Scenarios / Skenario Uji Cepat\n*Click any row to auto-fill the form. / Klik baris mana saja untuk mengisi formulir otomatis.*")
            gr.Examples(
                examples=[
                    # ── Grade I — Suspected Dengue, no labs, early febrile phase
                    # Suspek Dengue ringan, tanpa lab, fase febris awal
                    [18, "Male / Laki-laki",   2, 39.8, 6, 5, 0,       0.0,  0,    0, None],
                    # ── Grade II — Dengue Fever confirmed, mild thrombocytopenia
                    # Demam Dengue terkonfirmasi, trombositopenia ringan
                    [30, "Female / Perempuan", 4, 38.2, 5, 7, 120_000, 40.0, 4000, 1, None],
                    # ── Grade III — Dengue with Warning Signs, critical phase
                    # Dengue dengan Tanda Bahaya, fase kritis
                    [35, "Female / Perempuan", 5, 37.5, 3, 4, 45_000,  48.5, 3000, 2, None],
                    # ── Grade IV — Severe Dengue, life-threatening, multi-warning
                    # Dengue Berat, mengancam jiwa, banyak tanda bahaya
                    [45, "Male / Laki-laki",   6, 36.8, 2, 2, 12_000,  52.0, 2000, 4, None],
                ],
                inputs=[age, gender, day, fever, headache, myalgia,
                        platelets, hct, wbc, warnings_inp, img],
                label=(
                    "Grade I: Suspected · Grade II: Confirmed · "
                    "Grade III: Warning Signs · Grade IV: Severe"
                ),
            )

        # ── RIGHT: Output Dashboard ────────────────────────────────────────────
        with gr.Column(scale=5, elem_classes=["output-panel"]):
            gr.HTML(
                '<div class="section-label">🧠 Live Agent Reasoning / '
                'Penalaran Agen Langsung (Chain-of-Thought)</div>'
            )
            gr.Markdown(
                "*Each panel updates in real-time as the agent finishes. "
                "Results appear in English then Indonesian. / "
                "Setiap panel diperbarui secara real-time saat agen selesai. "
                "Hasil ditampilkan dalam Bahasa Inggris lalu Indonesia.*"
            )

            with gr.Accordion(
                "👁️ Agent 1 — Vision / Agen Visual  (Base MedGemma)", open=True
            ):
                out_vis = gr.Markdown(value=INIT_MSG)

            with gr.Accordion(
                "🩺 Agent 2 — Symptoms / Agen Gejala  (DeBe LoRA)", open=True
            ):
                out_sym = gr.Markdown(value=INIT_MSG)

            with gr.Accordion(
                "🧪 Agent 3 — Labs / Agen Lab  (DeBe LoRA)", open=True
            ):
                out_lab = gr.Markdown(value=INIT_MSG)

            gr.HTML(
                '<div class="section-label" style="margin-top:16px;">'
                "🏥 Orchestrator Final Decision / Keputusan Triase Akhir"
                "</div>"
            )
            out_final = gr.Markdown(value=INIT_MSG)

    # ── Footer ─────────────────────────────────────────────────────────────────
    gr.Markdown(
        """
---
**DeBe V3.0** • Built for the [MedGemma Impact Challenge](https://kaggle.com/competitions/med-gemma-impact-challenge) •
Model: [`arumpuri/medgemma-4b-debe-specialist`](https://huggingface.co/arumpuri/medgemma-4b-debe-specialist) •
*For research and demonstration purposes only / Hanya untuk keperluan riset dan demonstrasi*
        """
    )

    # ── Wire up ────────────────────────────────────────────────────────────────
    submit_btn.click(
        fn=gradio_wrapper,
        inputs=[age, gender, day, fever, headache, myalgia,
                platelets, hct, wbc, warnings_inp, img],
        outputs=[out_vis, out_sym, out_lab, out_final],
    )


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 10 — LAUNCH                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("\n[9/9] Launching DeBe Bilingual Clinical Dashboard...")
iface.launch(share=True, debug=False)


[9/9] Launching DeBe Bilingual Clinical Dashboard...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://99974be414fe52c0a1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
